In [45]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

# --------------------------------------------------
# Custom transformers
# --------------------------------------------------

class SelectiveStandardScaler(BaseEstimator, TransformerMixin):
    """
    Apply StandardScaler to selected columns only.
    All other columns are passed through unchanged.
    """
    def __init__(self, cols):
        self.cols = cols
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.cols])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = self.scaler.transform(X[self.cols])
        return X


def aggregate_by_observation(df):
    """
    Aggregate rows by `obs`:
    - Validate categorical columns are constant per obs
    - Average numeric columns
    """
    cat_cols = ["cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]

    # Ensure categorical consistency per observation
    inconsistent = (
        df.groupby("obs")[cat_cols]
          .nunique()
          .ne(1)
          .any()
    )
    if inconsistent.any():
        bad_cols = inconsistent[inconsistent].index.tolist()
        raise ValueError(f"Inconsistent categorical values per obs: {bad_cols}")

    numeric_cols = df.select_dtypes(include="number").columns
    agg = (
        df.groupby("obs", sort=False)[numeric_cols]
          .mean()
          .drop(columns="obs")
    )
    return agg


# --------------------------------------------------
# Preprocessing pipeline
# --------------------------------------------------

NUMERIC_SCALE_COLS = [
    "num_0", "num_1", "num_2",
    "t_0", "t_1", "t_2", "t_3", "t_4"
]

pipeline = Pipeline([
    (
        "coerce_numeric",
        FunctionTransformer(
            lambda df: df.apply(pd.to_numeric, errors="coerce"),
            validate=False
        )
    ),
    (
        "aggregate_obs",
        FunctionTransformer(aggregate_by_observation, validate=False)
    ),
    (
        "scale_selected",
        SelectiveStandardScaler(NUMERIC_SCALE_COLS)
    )
])


# --------------------------------------------------
# Run preprocessing
# --------------------------------------------------

df = pd.read_csv("Project_Data_export/train_competition_2026.csv")
df.sort_values("time", inplace=True)

# Preserve first timestamp per observation
time_by_obs = df.groupby("obs")["time"].first()

processed = pipeline.fit_transform(df)

# Reattach time and sort
processed["time"] = processed.index.map(time_by_obs)
processed.sort_values("time", inplace=True)

# Ensure categorical + id columns are ints
INT_COLS = ["sub_id", "cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]
processed[INT_COLS] = processed[INT_COLS].astype(int)

processed.head()


,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
obs,,,,,,,,,,,,,,,,,
9953,1021,2043-08-12 13:33:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.265702,0.232294,-0.425251,-0.702941,-0.448514,37.64,77.40
9954,1021,2043-08-12 18:20:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.192377,0.115235,-0.697081,-1.515308,-1.090151,27.98,76.82
9955,1021,2043-08-12 20:22:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.233228,0.125404,-0.084412,-0.996599,-0.460502,41.73,82.05
1513,132,2043-09-03 06:33:53,-1.968018,0.153046,0.679177,0,6,0,0,0,3.027694,0.327377,-0.208972,0.664390,0.211459,46.74,125.04
1514,132,2043-09-03 09:43:53,-1.968018,0.153046,0.679177,0,6,0,0,0,2.299069,0.140789,-0.422735,0.906359,0.283286,49.56,116.26
